# USAGE

## UPLOAD
1. Create new Kaggle notebook
2. File → Import Notebook → upload `kaggle_benchmark.ipynb`

## ENABLE GPU + INTERNET
1. Right panel → Settings
2. Accelerator → GPU T4 x2
3. Internet → On

## CONFIG (cell 10)

| Var | What | Default |
|-----|------|----------|
| `ACTIVE_DATASETS` | datasets to run | `["vivos", "commonvoice", "fosd", "infore", "vietmed", "vlsp"]` |
| `ACTIVE_MODELS` | models to run | `["faster-whisper-small", "phowhisper-small", "whisper-large", "phowhisper-large"]` |
| `LIMIT` | samples per dataset | `50` (None = full) |
| `MAX_WORKERS` | parallel threads | `2` |
| `THRESHOLD_CER` | weak case threshold | `0.25` |

## RUN
Cell → Run All (Ctrl+F9)

## DOWNLOAD
1. Output panel → benchmark_logs/ folder
2. Or download `benchmark_output.zip` (cell 12)

## OUTPUT FILES

| File | What |
|------|------|
| `benchmark_results.json` | CER/WER/RTF per model × dataset |
| `weak_cases_<ds>_<model>.jsonl` | samples with CER > 0.25 |

## MODIFY DATASETS

Available (from `backend/benchmark/datasets/loaders.py`):
```python
DATASETS = [
    {"name": "vivos", ...},
    {"name": "commonvoice", ...},
    {"name": "fosd", ...},
    {"name": "bud500", ...},  # large, ~500h
    {"name": "infore", ...},
    {"name": "vietmed", ...},
    {"name": "vlsp", ...},
]
```

## MODIFY MODELS

Available (from `config/models.yaml`):
```python
MODELS = [
    {"id": "faster-whisper-base", "engine": "faster-whisper", "model_size": "base", ...},
    {"id": "faster-whisper-small", "engine": "faster-whisper", "model_size": "small", ...},
    {"id": "phowhisper-small", "engine": "faster-whisper", "model_size": "mad1999/pho-whisper-small-ct2", ...},
    {"id": "phowhisper-medium", "engine": "phowhisper", "model_size": "medium"},
    {"id": "whisper-large", "engine": "faster-whisper", "model_size": "large-v3", ...},
    {"id": "phowhisper-large", "engine": "phowhisper", "model_size": "large"},
]
```

## TROUBLESHOOT

| Issue | Fix |
|-------|-----|
| OOM (VRAM) | reduce `MAX_WORKERS` to 1, or use `compute_type: int8` in MODELS |
| Timeout | reduce `LIMIT`, or run fewer models/datasets |
| Dataset load error | check HuggingFace dataset name, add `trust_remote_code: True` |
| Slow RTF | increase `MAX_WORKERS`, or use faster-whisper instead of transformers |


# Vietnamese ASR Benchmark - Kaggle Edition

Replicates the local benchmark (backend/benchmark) with parallel processing.

Supports both engines used locally:
- **faster-whisper** (CTranslate2) for Whisper base/small + PhoWhisper-small-ct2
- **transformers PhoWhisper** for PhoWhisper medium/large

Generates: results table, `benchmark_results.json`, and `weak_cases_*.jsonl` (CER > 0.25) per model/dataset.

In [1]:
# Install dependencies (transformers needed for PhoWhisper medium/large)
!pip install -q faster-whisper datasets tqdm transformers accelerate librosa

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 53.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 77.3 MB/s eta 0:00:00


In [2]:
import os
import json
import time
import re
import unicodedata
import numpy as np
from dataclasses import dataclass, asdict, field
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from datasets import load_dataset

LOG_DIR = "benchmark_logs"
os.makedirs(LOG_DIR, exist_ok=True)

DEVICE = "cuda"  # Kaggle GPU

In [3]:
# --- Vietnamese Text Normalization & Metrics (from backend/benchmark/metrics.py) ---

def normalize_vi(text: str) -> str:
    text = unicodedata.normalize("NFC", text.lower())
    text = re.sub(r"[^\w\sàáảãạăắằẳẵặâấầẩẫậèéẻẽẹêếềểễệ"
                  r"ìíỉĩịòóỏõọôốồổỗộơớờởỡợùúủũụưứừửữựỳýỷỹỵđ]",
                  " ", text)
    return re.sub(r"\s+", " ", text).strip()

def _edit_distance(ref, hyp):
    prev = list(range(len(hyp) + 1))
    for i, r in enumerate(ref, 1):
        cur = [i]
        for j, h in enumerate(hyp, 1):
            cur.append(prev[j - 1] if r == h else 1 + min(prev[j], cur[j - 1], prev[j - 1]))
        prev = cur
    return prev[-1]

def wer(reference, hypothesis):
    ref = normalize_vi(reference).split()
    hyp = normalize_vi(hypothesis).split()
    if not ref:
        return 0.0 if not hyp else 1.0
    return _edit_distance(ref, hyp) / len(ref)

def cer(reference, hypothesis):
    ref = list(normalize_vi(reference).replace(" ", ""))
    hyp = list(normalize_vi(hypothesis).replace(" ", ""))
    if not ref:
        return 0.0 if not hyp else 1.0
    return _edit_distance(ref, hyp) / len(ref)

def aggregate(refs, hyps):
    ref_words = hyp_word_err = 0
    ref_chars = char_err = 0
    for r, h in zip(refs, hyps):
        rw, hw = normalize_vi(r).split(), normalize_vi(h).split()
        ref_words += len(rw)
        hyp_word_err += _edit_distance(rw, hw)
        rc = list(normalize_vi(r).replace(" ", ""))
        hc = list(normalize_vi(h).replace(" ", ""))
        ref_chars += len(rc)
        char_err += _edit_distance(rc, hc)
    return {
        "wer": hyp_word_err / ref_words if ref_words else 0.0,
        "cer": char_err / ref_chars if ref_chars else 0.0,
    }

In [4]:
# --- Data Structures (from backend/benchmark/runner.py) ---

@dataclass
class ASRResult:
    model_id: str
    dataset: str
    n: int
    cer: float
    wer: float
    rtf: float
    failures: int = 0
    notes: list = field(default_factory=list)

    def row(self):
        n_label = f"{self.n}" + ("*" if self.failures else "")
        return (
            f"{self.model_id:<22} {self.dataset:<12} {n_label:>5} "
            f"{self.cer * 100:>6.1f} {self.wer * 100:>6.1f} {self.rtf:>6.2f}"
        )

@dataclass
class WeakCase:
    sample_id: int
    ref: str
    hyp: str
    cer: float
    confidence: float = None
    model_id: str = ""

In [5]:
# --- ASR Engines (ported from backend/adapters/asr) ---
import torch
import time

class FasterWhisperEngine:
    """CTranslate2 Whisper. Mirrors backend/adapters/asr/faster_whisper.py."""
    def __init__(self, model_size, compute_type="float16"):
        from faster_whisper import WhisperModel
        self._model = WhisperModel(model_size, device=DEVICE, compute_type=compute_type)
        
    def transcribe(self, samples, sample_rate):
        t0 = time.perf_counter()
        segments, info = self._model.transcribe(samples, language="vi")
        segs = [(s.text.strip(), getattr(s, "avg_logprob", None)) for s in segments]
        elapsed = time.perf_counter() - t0
        duration = len(samples) / sample_rate
        text = " ".join(t for t, _ in segs).strip()
        confs = [c for _, c in segs if c is not None]
        avg_conf = sum(confs) / len(confs) if confs else None
        rtf = (elapsed / duration) if duration else None
        return text, rtf, avg_conf

class PhoWhisperEngine:
    """transformers PhoWhisper chuẩn hóa mảng đầu vào và chặn thread-safe crash."""
    _MODEL_MAP = {
        "small": "vinai/PhoWhisper-small",
        "medium": "vinai/PhoWhisper-medium",
        "large": "vinai/PhoWhisper-large",
    }
    def __init__(self, model_size):
        from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
        import os
        
        # Tắt telemetry Hub để tránh lỗi 403 Forbidden luồng ngầm
        os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
        os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
        
        hf_name = self._MODEL_MAP.get(model_size, model_size)
        
        self.processor = AutoProcessor.from_pretrained(hf_name)
        self.model = AutoModelForSpeechSeq2Seq.from_pretrained(
            hf_name,
            tie_word_embeddings=False,
            torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
        ).to(DEVICE)
        
        self._pipe = pipeline(
            "automatic-speech-recognition",
            model=self.model,
            tokenizer=self.processor.tokenizer,
            feature_extractor=self.processor.feature_extractor,
            device=0 if DEVICE == "cuda" else -1,
        )
        
    def transcribe(self, samples, sample_rate):
        import numpy as np
        t0 = time.perf_counter()
        
        # Đảm bảo samples ở dạng định dạng chuẩn numpy 1D float32
        audio_array = np.asarray(samples, dtype=np.float32)
        
        # Thực hiện nhận diện với các tham số ép giải mã tiếng Việt ổn định
        out = self._pipe(
            audio_array,
            chunk_length_s=30,
            stride_length_s=5,
            generate_kwargs={"language": "vietnamese", "task": "transcribe"}
        )
        
        elapsed = time.perf_counter() - t0
        duration = len(samples) / sample_rate
        rtf = (elapsed / duration) if duration else None
        return out["text"].strip(), rtf, None

def build_engine(model_cfg):
    if model_cfg["engine"] == "faster-whisper":
        return FasterWhisperEngine(model_cfg["model_size"], model_cfg.get("compute_type", "float16"))
    elif model_cfg["engine"] == "phowhisper":
        return PhoWhisperEngine(model_cfg["model_size"])
    raise ValueError(f"unknown engine {model_cfg['engine']}")

In [6]:
def load_samples(dataset_config, limit = None):
    print(f"Loading dataset: {dataset_config['path']} ...")
    ds = load_dataset(
        dataset_config["path"], 
        dataset_config.get("hf_name"), 
        split=dataset_config["split"], 
        trust_remote_code=dataset_config.get("trust_remote_code", False)
    )
    if limit:
        ds = ds.select(range(min(limit, len(ds))))
        
    samples = []
    for sample in ds:
        audio_data = sample["audio"]
        
        # Handle the new torchcodec AudioDecoder objects
        if hasattr(audio_data, "get_all_samples"):
            decoded = audio_data.get_all_samples()
            audio = decoded.data.numpy().astype(np.float32)
            # Flatten to 1D array if multi-channel/stereo
            if audio.ndim > 1:
                audio = audio.ravel()
            sr = decoded.sample_rate
        # Handle legacy dictionary format
        else:
            audio = audio_data["array"].astype(np.float32)
            sr = audio_data.get("sampling_rate", 16000)
            
        ref_text = sample[dataset_config["text_col"]]
        samples.append((audio, sr, ref_text))
        
    return samples

In [7]:
# --- Parallel Benchmark Runner (mirrors backend/benchmark/runner.py run_asr) ---

def run_asr_parallel(model_cfg, dataset_name, samples, threshold_cer=0.25, max_workers=2):
    model_id = model_cfg["id"]
    print(f"\nLoading model: {model_id} ({model_cfg['engine']})...")
    engine = build_engine(model_cfg)

    def process_sample(idx, audio, sr, ref):
        try:
            text, rtf, conf = engine.transcribe(audio, sr)
            return (idx, ref, text, rtf, conf, None)
        except Exception as e:
            return (idx, ref, "", None, None, str(e))

    results = [None] * len(samples)
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(process_sample, i, audio, sr, ref)
            for i, (audio, sr, ref) in enumerate(samples)
        ]
        progress = tqdm(total=len(samples), desc=f"{dataset_name[:8]} | {model_id[:15]}", leave=False, ncols=80)
        for future in as_completed(futures):
            idx, ref, hyp, rtf, conf, error = future.result()
            results[idx] = (ref, hyp, rtf, conf, error)
            progress.update(1)
        progress.close()

    refs, hyps, rtfs, failures = [], [], [], 0
    weak_cases = []
    for i, (ref, hyp, rtf, conf, error) in enumerate(results):
        if error:
            failures += 1
            continue
        refs.append(ref)
        hyps.append(hyp)
        if rtf is not None:
            rtfs.append(rtf)
        sample_cer = cer(ref, hyp)
        if sample_cer > threshold_cer:
            weak_cases.append(WeakCase(sample_id=i, ref=ref, hyp=hyp, cer=sample_cer, confidence=conf, model_id=model_id))

    scores = aggregate(refs, hyps) if refs else {"cer": 0.0, "wer": 0.0}

    weak_cases.sort(key=lambda x: x.cer, reverse=True)
    worst_cases = weak_cases[:50]

    log_file = os.path.join(LOG_DIR, f"weak_cases_{dataset_name}_{model_id.replace('/', '_')}.jsonl")
    with open(log_file, "w", encoding="utf-8") as f:
        if worst_cases:
            for case in worst_cases:
                f.write(json.dumps(asdict(case), ensure_ascii=False) + "\n")
        else:
            f.write(json.dumps({
                "summary": f"No high-error cases found (CER > {threshold_cer}) in {len(refs)} samples",
                "dataset": dataset_name,
                "model_id": model_id,
                "samples_tested": len(refs),
                "avg_cer": scores["cer"],
            }, ensure_ascii=False) + "\n")

    return ASRResult(
        model_id=model_id,
        dataset=dataset_name,
        n=len(refs),
        cer=scores["cer"],
        wer=scores["wer"],
        rtf=sum(rtfs) / len(rtfs) if rtfs else 0.0,
        failures=failures,
    )

In [8]:
# --- Configuration ---

# Datasets (from backend/benchmark/datasets/loaders.py)
DATASETS = [
    {"name": "vivos", "path": "AILAB-VNUHCM/vivos", "split": "test", "text_col": "sentence", "trust_remote_code": True},
    {"name": "commonvoice", "path": "fsicoli/common_voice_17_0", "hf_name": "vi", "split": "test", "text_col": "sentence", "trust_remote_code": True},
    {"name": "fosd", "path": "doof-ferb/fpt_fosd", "split": "train", "text_col": "transcription"},
    {"name": "bud500", "path": "linhtran92/viet_bud500", "split": "test", "text_col": "transcription"},
    {"name": "infore", "path": "doof-ferb/infore1_25hours", "split": "train", "text_col": "transcription"},
    {"name": "vietmed", "path": "leduckhai/VietMed", "split": "test", "text_col": "text"},
    {"name": "vlsp", "path": "doof-ferb/vlsp2020_vinai_100h", "split": "train", "text_col": "transcription"},
    {"name": "vimedcss", "path": "tensorxt/ViMedCSS", "split": "test", "text_col": "text"},
]
# Models (from config/models.yaml). compute_type=float16 for Kaggle GPU (local uses int8 on CPU).
MODELS = [
    {"id": "faster-whisper-base", "engine": "faster-whisper", "model_size": "base", "compute_type": "float16"},
    {"id": "faster-whisper-small", "engine": "faster-whisper", "model_size": "small", "compute_type": "float16"},
    {"id": "phowhisper-small", "engine": "faster-whisper", "model_size": "mad1999/pho-whisper-small-ct2", "compute_type": "float16"},
    {"id": "phowhisper-medium", "engine": "phowhisper", "model_size": "medium"},
    {"id": "whisper-large", "engine": "faster-whisper", "model_size": "large-v3", "compute_type": "float16"},
    {"id": "phowhisper-large", "engine": "phowhisper", "model_size": "large"},
]

# Pick subsets to run (comment out what you don't need)
ACTIVE_DATASETS = [ "infore"]  # bud500 is large
ACTIVE_MODELS = ["phowhisper-large"]

LIMIT = None       # samples per dataset (None = full)
MAX_WORKERS = 1     # parallel workers
THRESHOLD_CER = 0.3

In [9]:
import os
from kaggle_secrets import UserSecretsClient
# faster dowloadn on datasets
try:
    user_secrets = UserSecretsClient()
    # Lấy token từ Secrets và set vào biến môi trường
    os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
    print("✅ Đã cấu hình HF_TOKEN thành công từ Kaggle Secrets!")
except Exception as e:
    print("❌ Không tìm thấy Secrets. Hãy đảm bảo bạn đã tạo Secret với nhãn 'HF_TOKEN'.")

✅ Đã cấu hình HF_TOKEN thành công từ Kaggle Secrets!


In [10]:
# --- Run Benchmark ---

datasets_to_run = [d for d in DATASETS if d["name"] in ACTIVE_DATASETS]
models_to_run = [m for m in MODELS if m["id"] in ACTIVE_MODELS]

all_results = []
for ds_config in datasets_to_run:
    try:
        samples = load_samples(ds_config, limit=LIMIT)
        for model_cfg in models_to_run:
            try:
                result = run_asr_parallel(model_cfg, ds_config["name"], samples, THRESHOLD_CER, MAX_WORKERS)
                all_results.append(result)
            except Exception as e:
                print(f"Error with model {model_cfg['id']} on {ds_config['name']}: {e}")
    except Exception as e:
        print(f"Error loading dataset {ds_config['name']}: {e}")

Loading dataset: doof-ferb/infore1_25hours ...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00015.parquet:   0%|          | 0.00/486M [00:00<?, ?B/s]

data/train-00001-of-00015.parquet:   0%|          | 0.00/485M [00:00<?, ?B/s]

data/train-00002-of-00015.parquet:   0%|          | 0.00/522M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/1260 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


infore | phowhisper-larg:   0%|                       | 0/14935 [00:00<?, ?it/s]

A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
Using `chunk_length_s` is very 

In [11]:
# --- Format and Display Results (mirrors backend/benchmark/runner.py format_table) ---

def format_table(results, limit):
    header = f"{'model':<22} {'dataset':<12} {'n':>5} {'CER%':>6} {'WER%':>6} {'RTF':>6}"
    lines = [
        f"ASR benchmark  (limit={limit if limit is not None else 'full'}, CER is the headline metric for Vietnamese)",
        header,
        "-" * len(header),
    ]
    lines += [r.row() for r in results]
    if any(r.failures for r in results):
        lines.append("\n* n marked with * had transcription failures (see logs).")
    return "\n".join(lines)

print("\n" + "=" * 80)
print(format_table(all_results, LIMIT))
print("=" * 80)

results_file = os.path.join(LOG_DIR, "benchmark_results.json")
with open(results_file, "w", encoding="utf-8") as f:
    json.dump([asdict(r) for r in all_results], f, ensure_ascii=False, indent=2)

print(f"\nSaved to {LOG_DIR}/")
print("- benchmark_results.json")
print("- weak_cases_<dataset>_<model>.jsonl (one per model/dataset)")


ASR benchmark  (limit=full, CER is the headline metric for Vietnamese)
model                  dataset          n   CER%   WER%    RTF
--------------------------------------------------------------
phowhisper-large       infore       14935   58.5   74.8   0.46

Saved to benchmark_logs/
- benchmark_results.json
- weak_cases_<dataset>_<model>.jsonl (one per model/dataset)


In [12]:
# --- Zip all outputs for easy download ---
import shutil
shutil.make_archive("benchmark_output", "zip", LOG_DIR)
print("Created benchmark_output.zip - download from the Kaggle output panel.")

Created benchmark_output.zip - download from the Kaggle output panel.
